# 🎙️ ZeroTTS Studio (TSS) - Kaggle GPU (Free Tier 30h/tuần)
**Mô hình Text-to-Speech (TTS) Tiếng Việt với toàn bộ tính năng: Phân đoạn tag `[Câu 1]`, `#[Bỏ qua]`, `[pause: 1.5s]`, gộp `_FULL_MERGED.mp3`, Streaming Audio.**

🔗 Mã nguồn GitHub: [RevenantKitana/TSS](https://github.com/RevenantKitana/TSS)

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code)

---

### ⚡ 2 Thiết Lập Bắt Buộc Trước Khi Chạy Trên Kaggle:
Nhìn sang cột bên phải (**Notebook settings**):
1. **Accelerator**: Chọn **GPU T4 x2** (hoặc **GPU P100**) để kích hoạt tăng tốc phần cứng.
2. **Internet**: Bật **Internet -> ON** (Bắt buộc gạt nút sang bật để tải mã nguồn, mô hình và mở Cloudflare Tunnel).

> 💡 *Nếu bạn gặp thông báo: `Error: Permission 'kernelSessions.enableInternet' was denied`*
> *👉 Hãy vào [kaggle.com/settings](https://www.kaggle.com/settings) -> Kéo xuống **Phone Verification** để xác minh số điện thoại 1 lần duy nhất. Sau đó bạn sẽ có 30h GPU/tuần miễn phí!*
---

### 🎯 Quy trình chạy nhanh (1-Click Run All):
- **Bước 1**: Kiểm tra GPU & Môi trường Kaggle.
- **Bước 2**: Tự động tải mã nguồn trực tiếp từ GitHub [RevenantKitana/TSS](https://github.com/RevenantKitana/TSS).
- **Bước 3**: Cài đặt thư viện & ONNX Runtime GPU (CUDA).
- **Bước 4**: Tải Model Weights từ Hugging Face.
- **Bước 5**: Nhấp vào link **Public HTTPS URL** (`https://xxxx.trycloudflare.com`) để mở WebUI Studio!

## ⚙️ Bước 1: Kiểm tra GPU & Kết Nối Mạng (Kaggle Environment)
> *Kiểm tra xem GPU T4/P100 đã được bật và Internet đã mở hay chưa.*

In [ ]:
#@title Kiểm tra Phần Cứng & Kết Nối Internet
import os
import sys
import subprocess
import urllib.request

print("🔍 1. Đang kiểm tra GPU...")
HAS_GPU = False
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode("utf-8").strip()
    print(f"   ✅ Phát hiện GPU: {gpu_info}")
    HAS_GPU = True
except Exception:
    print("   ⚠️ Đang chạy trên CPU! Vui lòng vào thanh bên phải: Settings -> Accelerator -> Chọn GPU T4 x2 hoặc GPU P100.")

print("\n🌐 2. Đang kiểm tra kết nối Internet...")
try:
    urllib.request.urlopen("https://github.com", timeout=5)
    print("   ✅ Kết nối Internet đã BẬT (Internet: ON).")
except Exception:
    print("   ❌ INTERNET ĐANG TẮT! Vui lòng vào thanh bên phải: Settings -> Bật 'Internet' sang ON rồi chạy lại cell này.")
    print("   (Lưu ý: Cần xác minh SĐT tại kaggle.com/settings nếu bị báo lỗi Permission)")

## 📦 Bước 2: Nạp Mã Nguồn Trực Tiếp Từ GitHub
> *Tự động clone phiên bản mới nhất từ [https://github.com/RevenantKitana/TSS](https://github.com/RevenantKitana/TSS) — Không cần upload file zip thủ công.*

In [ ]:
#@title Nạp Mã Nguồn Từ GitHub (RevenantKitana/TSS)
import os
import sys
import subprocess
import shutil

REPO_URL = "https://github.com/RevenantKitana/TSS.git"
APP_DIR = "/kaggle/working/TSS"
OUTPUT_DIR = "/kaggle/working/outputs"
os.environ["ZEROTTS_OUTPUT_DIR"] = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📥 Đang nạp mã nguồn từ GitHub: {REPO_URL}...")
if not os.path.exists(os.path.join(APP_DIR, ".git")):
    shutil.rmtree(APP_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, APP_DIR], check=True)
    print("✅ Đã clone mã nguồn thành công!")
else:
    print("🔄 Cập nhật mã nguồn mới nhất từ GitHub...")
    subprocess.run(["git", "-C", APP_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", APP_DIR, "reset", "--hard", "origin/main"], check=True)
    print("✅ Đã đồng bộ mã nguồn mới nhất!")

# Đồng bộ thư mục webui và src vào /kaggle/working để đảm bảo tương thích mọi đường dẫn
for folder in ["webui", "src"]:
    src_f = os.path.join(APP_DIR, folder)
    dst_f = os.path.join("/kaggle/working", folder)
    if os.path.exists(src_f) and not os.path.exists(dst_f):
        try:
            shutil.copytree(src_f, dst_f, dirs_exist_ok=True)
        except Exception:
            pass

if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

print(f"\n📂 Thư mục mã nguồn: {APP_DIR}")
print(f"📁 Thư mục lưu audio: {OUTPUT_DIR}")
print("✅ Sẵn sàng bước tiếp theo!")

## 🔧 Bước 3: Cài Đặt Thư Viện & ONNX Runtime Tăng Tốc GPU
> *Cài đặt ffmpeg, libportaudio, ONNX Runtime GPU (hỗ trợ CUDA) và các package cần thiết.*

In [ ]:
#@title Cài Đặt Thư Viện (Chỉ mất ~1 phút)
import os
import sys

APP_DIR = "/kaggle/working/TSS"
work_dir = APP_DIR if os.path.exists(APP_DIR) else "/kaggle/working"
os.chdir(work_dir)
print(f"📍 Thư mục làm việc: {os.getcwd()}")

print("🔧 1. Đang cài đặt thư viện hệ thống (ffmpeg, libportaudio)...\n")
!apt-get update -qq && apt-get install -y -qq ffmpeg libportaudio2

print("\n⚡ 2. Đang cài đặt ONNX Runtime GPU & Python dependencies...")
if HAS_GPU:
    !pip install -q "onnxruntime-gpu>=1.17.0"
else:
    !pip install -q "onnxruntime>=1.17.0"

!pip install -q soundfile sounddevice fastapi uvicorn tokenizers huggingface_hub scipy requests pydantic

if os.path.exists("pyproject.toml") or os.path.exists("setup.py"):
    !pip install -q -e .

print("\n✅ Cài đặt hoàn tất! Môi trường đã sẵn sàng hoạt động với đầy đủ tính năng.")

## 🧠 Bước 4: Tải Model Weights (ZeroTTS từ Hugging Face)
> *Tự động tải trọng số mô hình `zeroweight-ai/ZeroTTS` (~500MB) về máy chủ.*

In [ ]:
#@title Tải Model Weights
from huggingface_hub import snapshot_download
import os
import glob

APP_DIR = "/kaggle/working/TSS"
MODEL_DIR = os.path.join(APP_DIR, "ZeroTTS_model") if os.path.exists(APP_DIR) else "/kaggle/working/ZeroTTS_model"
os.environ["ZEROTTS_MODEL"] = MODEL_DIR

# Kiểm tra nếu model đã được đính kèm qua Kaggle Dataset
preloaded_models = glob.glob("/kaggle/input/**/model.onnx", recursive=True)
if preloaded_models:
    MODEL_DIR = os.path.dirname(preloaded_models[0])
    os.environ["ZEROTTS_MODEL"] = MODEL_DIR
    print(f"✅ Sử dụng mô hình có sẵn từ Kaggle Dataset: {MODEL_DIR}")
elif not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("⏳ Đang tải mô hình ZeroTTS từ Hugging Face (~500MB)... Vui lòng chờ 15-30 giây.")
    snapshot_download(repo_id="zeroweight-ai/ZeroTTS", local_dir=MODEL_DIR)
    print(f"✅ Đã tải xong Model Weights vào {MODEL_DIR}!")
else:
    print(f"✅ Mô hình đã sẵn sàng tại {MODEL_DIR}!")

## 🚀 Bước 5: Khởi Chạy WebUI Studio Kèm Cloudflare Tunnel
> *Cloudflare Tunnel sẽ cung cấp một đường link Public HTTPS an toàn (`https://xxxx.trycloudflare.com`) để bạn truy cập WebUI ngay trên trình duyệt mà không cần cài đặt thêm bất kỳ phần mềm nào.*

In [ ]:
#@title Khởi chạy WebUI Studio & Mở Public HTTPS URL
import subprocess
import time
import re
import os
import sys
import glob
import shutil
from IPython.display import display, HTML

# 1. Tự động xác định chính xác vị trí webui/server.py và thư mục gốc dự án
APP_DIR = "/kaggle/working/TSS"
server_candidates = [
    os.path.join(APP_DIR, "webui", "server.py"),
    "/kaggle/working/TSS/webui/server.py",
    "/kaggle/working/webui/server.py",
    os.path.join(os.getcwd(), "webui", "server.py"),
]
SERVER_SCRIPT = None
PROJECT_ROOT = None
for cand in server_candidates:
    if os.path.isfile(cand):
        SERVER_SCRIPT = os.path.abspath(cand)
        PROJECT_ROOT = os.path.dirname(os.path.dirname(SERVER_SCRIPT))
        break

if not SERVER_SCRIPT:
    matches = glob.glob("/kaggle/**/webui/server.py", recursive=True) + glob.glob("**/webui/server.py", recursive=True)
    if matches:
        SERVER_SCRIPT = os.path.abspath(matches[0])
        PROJECT_ROOT = os.path.dirname(os.path.dirname(SERVER_SCRIPT))

if not SERVER_SCRIPT or not os.path.isfile(SERVER_SCRIPT):
    raise FileNotFoundError("❌ Không tìm thấy file webui/server.py! Hãy chạy lại Bước 2 (Nạp mã nguồn từ GitHub).")

print(f"🎯 Đã định vị WebUI Server tại: {SERVER_SCRIPT}")
print(f"📂 Project Root: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)

# 2. Download binary cloudflared nếu chưa có
CF_BIN = "/kaggle/working/cloudflared"
if not os.path.exists(CF_BIN):
    print("📥 Đang tải Cloudflare Tunnel binary (cloudflared)...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
    !chmod +x /kaggle/working/cloudflared

# 3. Khởi chạy FastAPI Server
print("🚀 1. Đang khởi động ZeroTTS WebUI Server...")
server_cmd = [sys.executable, SERVER_SCRIPT, "--model", MODEL_DIR, "--host", "0.0.0.0", "--port", "7860"]
server_proc = subprocess.Popen(
    server_cmd,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

time.sleep(3)

# 4. Khởi động Cloudflare Tunnel
print("🌐 2. Đang thiết lập đường hầm Cloudflare Tunnel...")
tunnel_cmd = [CF_BIN, "tunnel", "--url", "http://127.0.0.1:7860"]
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for _ in range(50):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.4)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("=" * 70)
if public_url:
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #1e1e2e, #2d1f47); padding: 20px; border-radius: 12px; border: 1px solid #7c3aed; margin: 15px 0;">
        <h2 style="color: #a78bfa; margin: 0 0 10px 0;">🎉 ZeroTTS Studio Đã Sẵn Sàng!</h2>
        <p style="color: #e2e8f0; font-size: 15px; margin: 0 0 15px 0;">Nhấp vào nút dưới đây để mở giao diện WebUI trực tiếp trên trình duyệt của bạn:</p>
        <a href="{public_url}" target="_blank" style="background: #7c3aed; color: #ffffff; padding: 12px 24px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 16px; display: inline-block; box-shadow: 0 4px 14px rgba(124, 58, 237, 0.4);">
            🚀 Mở WebUI Studio (Public URL)
        </a>
        <p style="color: #94a3b8; font-size: 13px; margin: 12px 0 0 0;">Link: <a href="{public_url}" target="_blank" style="color: #38bdf8;">{public_url}</a></p>
    </div>
    """))
else:
    print("⚠️ Đang chờ đường hầm kết nối, vui lòng theo dõi log...")
print("=" * 70 + "\n")

# Hiển thị log liên tục từ server
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="", flush=True)
        else:
            time.sleep(0.1)
except KeyboardInterrupt:
    print("\n🛑 Đang dừng server và đóng tunnel...")
    server_proc.terminate()
    tunnel_proc.terminate()
    print("✅ Đã dừng hoàn toàn.")

## ⚡ Bước 6 (Tùy Chọn): Chế Độ Dòng Lệnh (CLI Batch Synthesis)
> *Sinh âm thanh hàng loạt trực tiếp trong Notebook với đầy đủ cú pháp tag `[Câu 1]`, `#[Bỏ qua]`, `[pause: 1.5s]`, `[Kết Thúc]` và tự động gộp file `_FULL_MERGED.mp3` mà không cần mở WebUI.*

In [ ]:
#@title Chạy Render Hàng Loạt Trực Tiếp Bằng Code
sample_input_text = """[Câu 1]
Xin chào các bạn. [pause: 1.5s] Đây là câu hỏi trắc nghiệm đầu tiên.

[Câu 2]
Hãy chọn một phương án chính xác nhất trong 4 phương án sau đây.

#[Bỏ qua]
Đoạn ghi chú nội bộ này sẽ được hệ thống bỏ qua không đọc.

[Kết Thúc]
Chúc các bạn hoàn thành bài thi thật tốt và đạt kết quả cao!
"""

VOICE_NAME = "nam-mien-bac" #@param ["nam-mien-bac", "nu-mien-bac", "nam-mien-nam", "nu-mien-nam", "unconditional"]
PROJECT_NAME = "du_an_kaggle_01" #@param {type:"string"}

if PROJECT_ROOT and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import webui.engine as engine

engine.set_model(MODEL_DIR)

print(f"🎯 Đang xử lý kịch bản cho dự án: {PROJECT_NAME}...")
blocks = engine.parse_input_tags(sample_input_text)

print(f"📋 Tổng số phân đoạn nhận diện: {len(blocks)}")
for b in blocks:
    status = "⏭️ BỎ QUA" if b["is_skipped"] else "✅ RENDER"
    print(f" - [{b['tag']}]: {status} | Nội dung: {b['text'][:40]}...")

voice_param = None if VOICE_NAME == "unconditional" else VOICE_NAME
mode_param = "uncond" if VOICE_NAME == "unconditional" else "voice"

print("\n🔊 Đang tiến hành sinh âm thanh và gộp file...")
out_folder = None
for event_data in engine.generate_from_parsed_blocks(
    blocks=blocks,
    voice_name=voice_param,
    mode=mode_param,
    custom_name=PROJECT_NAME,
    auto_concat=True,
    merged_format="MP3"
):
    if "status" in event_data:
        print(f"   {event_data['status']}")
    if "out_folder" in event_data:
        out_folder = event_data["out_folder"]

print(f"\n🎉 Hoàn thành! Thư mục kết quả: {out_folder}")
if out_folder and os.path.exists(out_folder):
    print("📂 Danh sách file đã tạo:")
    for f in sorted(os.listdir(out_folder)):
        fpath = os.path.join(out_folder, f)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  ├── {f} ({size_kb:.1f} KB)")

## 💾 Bước 7: Đóng Gói & Tải Xuống Toàn Bộ File Âm Thanh (Download Outputs)
> *Gom toàn bộ file `.mp3`, `.wav`, timeline `.json` thành 1 file zip để tải về máy tính chỉ với 1 cú click.*

In [ ]:
#@title Nén Thư Mục Outputs & Tạo Link Tải Về Máy
import os
import shutil
from IPython.display import display, FileLink, HTML

ZIP_OUTPUT_NAME = "/kaggle/working/ZeroTTS_Outputs.zip"
target_folder = os.environ.get("ZEROTTS_OUTPUT_DIR", "/kaggle/working/outputs")

if os.path.exists(target_folder) and os.listdir(target_folder):
    print(f"📦 Đang nén thư mục {target_folder} thành {ZIP_OUTPUT_NAME}...")
    shutil.make_archive("/kaggle/working/ZeroTTS_Outputs", 'zip', target_folder)
    zip_size_mb = os.path.getsize(ZIP_OUTPUT_NAME) / (1024 * 1024)
    print(f"✅ Đã tạo file zip thành công ({zip_size_mb:.2f} MB)!")
    
    display(HTML(f"""
    <div style="background: #1e293b; padding: 15px; border-radius: 8px; border: 1px solid #38bdf8; margin-top: 10px;">
        <h3 style="color: #38bdf8; margin: 0 0 8px 0;">📥 Tải Toàn Bộ Kết Quả Về Máy:</h3>
        <a href="ZeroTTS_Outputs.zip" download style="color: #f8fafc; background: #0284c7; padding: 8px 16px; border-radius: 6px; text-decoration: none; font-weight: bold; display: inline-block;">
            ⬇️ Tải xuống ZeroTTS_Outputs.zip ({zip_size_mb:.2f} MB)
        </a>
        <p style="color: #94a3b8; font-size: 13px; margin: 8px 0 0 0;">Hoặc bạn có thể tải từng file tại thẻ <b>Data / Output</b> ở cột bên phải giao diện Kaggle.</p>
    </div>
    """))
else:
    print(f"⚠️ Thư mục {target_folder} hiện đang trống hoặc chưa có file render nào.")